In [ ]:
!pip install tensorflow scikit-learn numpy pandas

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
df = pd.read_csv("/content/GNSS_Merged_Dataset.csv")

print(df.shape)
print(df.head())

(6000, 16)
   Time(ms)    Temp(C)  Humidity(%)  RainAnalog  RainDigital  SNR1  SNR2  \
0    178200  33.691820    40.216222          60            0    49    50   
1    191700  25.062289    65.934869         317            1    42    41   
2     22100  34.546469    50.813970          61            0    52    49   
3     13500  30.950958    63.467639         440            1    41    39   
4    195000  22.264200    96.614697         854            1    35    34   

   SNR3  SNR4  SNR5  SNR6  SNR7   Mean_SNR    SNR_Var  Sat_Count  WeatherLabel  
0    49    50    49    50    49  49.428571   0.285714          7             0  
1    43    39    42    41    43  42.125000   4.125000          8             1  
2    48    52    50    51    51  50.428571   2.285714          7             0  
3    39    43    39    43    39  41.125000   6.982143          8             1  
4    33    32    33    32    36  31.000000  87.000000         11             2  


In [ ]:
feature_columns = [f"SNR{i}" for i in range(1,7)]

X = df[feature_columns].values
y = df["WeatherLabel"].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
mean = scaler.mean_
std = scaler.scale_

np.save("scaler_mean.npy", mean)
np.save("scaler_std.npy", std)

print("Mean values:\n", mean)
print("Std values:\n", std)

Mean values:
 [41.3375     41.33666667 41.38270833 41.27833333 41.37666667 41.271875  ]
Std values:
 [7.08641967 7.15180552 7.04571331 7.10270117 7.14567157 7.24169356]


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(12, activation="relu", input_shape=(6,)),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(3, activation="softmax")
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1
)

Epoch 1/50
135/135 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.6462 - loss: 0.7049 - val_accuracy: 0.8375 - val_loss: 0.3370
Epoch 2/50
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8879 - loss: 0.3229 - val_accuracy: 0.9875 - val_loss: 0.1923
Epoch 3/50
135/135 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9993 - loss: 0.1752 - val_accuracy: 1.0000 - val_loss: 0.0821
Epoch 4/50
135/135 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0686 - val_accuracy: 1.0000 - val_loss: 0.0292
Epoch 5/50
135/135 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0241 - val_accuracy: 1.0000 - val_loss: 0.0120
Epoch 6/50
135/135 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0102 - val_accuracy: 1.0000 - val_loss: 0.0061
Epoch 7/50
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 1.0000 - loss: 0.0053 - val_accuracy: 1.0000 - val_loss: 0.0035
Epoch 8/50
135/135 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0031 - val_accuracy: 1.

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)
y_pred = np.argmax(model.predict(X_test), axis=1)
print(classification_report(y_test, y_pred))

38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 1.0000 - loss: 8.2790e-06
Test Accuracy: 1.0
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       400
           1       1.00      1.00      1.00       400
           2       1.00      1.00      1.00       400

    accuracy                           1.00      1200
   macro avg       1.00      1.00      1.00      1200
weighted avg       1.00      1.00      1.00      1200



In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open("rain_model.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model saved successfully.")

Saved artifact at '/tmp/tmpu9ozpqrn'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 6), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  136695043308432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136695043309392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136695043306128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136695043307280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136695043308240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136695043307088: TensorSpec(shape=(), dtype=tf.resource, name=None)
TFLite model saved successfully.


In [ ]:
!apt-get install xxd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
xxd is already the newest version (2:8.2.3995-1ubuntu2.24).
xxd set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [ ]:
!xxd -i rain_model.tflite > rain_model.cc

In [ ]:
!ls -lh rain_model.cc

-rw-r--r-- 1 root root 19K Feb 18 19:51 rain_model.cc


In [ ]:
!xxd -i rain_model.tflite > model_data.cpp

In [ ]:
for layer in model.layers:
    weights = layer.get_weights()
    print(weights)

[array([[ 0.11696525,  0.0535046 ,  0.36468148, -0.85573214,  0.6339032 ,
         0.45372412, -0.60291517, -0.09457244, -0.02383789, -0.5931985 ,
        -0.44790736,  0.6489324 ],
       [-0.2426535 ,  0.14009131,  0.5352708 , -0.79854375,  0.16499846,
        -0.6095386 , -0.16505763,  0.49067828, -0.43403608, -0.8151489 ,
         0.1127146 ,  0.55518174],
       [-0.57290214,  0.12685639,  0.82228297, -0.36502123,  0.42967993,
         0.20968555,  0.35873955,  0.47687316, -0.62269264, -0.2739105 ,
        -0.3883196 , -0.35184163],
       [-0.7430301 ,  0.63666064,  0.8009172 ,  0.13024016, -0.40630504,
         0.21975449,  0.3514523 ,  0.55423754, -0.5764871 , -0.42006877,
         0.15687464,  0.46965086],
       [-0.5320567 , -0.00287538, -0.21645008, -0.0439676 , -0.23807968,
        -0.29072386,  0.1032264 ,  0.84935325, -0.4313498 , -0.23971213,
        -0.0171913 ,  0.02537679],
       [ 0.10409871,  0.48484397, -0.06991861, -0.15758431, -0.133153  ,
        -0.27603763, 